> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Investment & Technology Value Realization (TVR)**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../09_investment_benefit_realization.ipynb`.

# Notebook 09: Investment & Technology Value Realization (TVR)

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 09 of 12  
**Focus:** Project Budget Variance, Expected vs Realized Benefits, Benefit Realization Gap, KPI Outcomes  

---

## 1. Business Problem

Enterprises spend billions annually on digital transformations, cloud migrations, and software modernization projects. Yet, industry studies consistently report that over 65% of enterprise technology projects fail to deliver their anticipated financial or operational returns.

The root cause is a post-delivery accountability vacuum:
1. When a project is delivered, finance records the capital expenditure, but **stops tracking whether the promised benefits actually materialize**.
2. Project teams disband and claim success based purely on delivery date, while business sponsors never verify whether productivity or cost reduction targets were met.

**Technology Value Realization (TVR)** establishes a continuous feedback loop connecting capital investment to tangible outcome realization.

---

## 2. Core Concept & Formulation

### 2.1 Investment Budget Variance
$$\text{Investment Variance} = \text{Actual Spend} - \text{Investment Budget}$$
$$\text{Budget Variance \%} = \frac{\text{Actual Spend} - \text{Budget}}{\text{Budget}} \times 100$$

### 2.2 Benefit Realization & Gap Metrics
$$\text{Benefit Realization \%} = \frac{\text{Recorded Realized Benefit}}{\text{Expected Annual Benefit}} \times 100$$
$$\text{Benefit Gap} = \text{Expected Annual Benefit} - \text{Recorded Realized Benefit}$$

### 2.3 Strict Analytical Guardrail: Association vs Causality
We explicitly enforce consulting guardrails:  
*We state that a project is "associated with" a benefit or KPI movement, rather than claiming mathematical causation, unless causal experiments (A/B testing, synthetic controls) were formally conducted.*

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config
from tvi.value_realization import (
    calculate_project_budget_variance, calculate_benefit_realization,
    analyze_benefit_kpi_progression, get_project_value_profile
)
from tvi.database import get_database

config = load_config()
db = get_database(config.database.path)

# Calculate budget variances and benefit realization rates
budget_df = calculate_project_budget_variance()
benefit_df = calculate_benefit_realization()

tvr_df = budget_df.merge(benefit_df[["project_id", "total_expected_benefit", "total_realized_benefit", "benefit_realization_pct", "benefit_gap"]], on="project_id")
print("Technology Value Realization Portfolio Summary:")
display(tvr_df[["project_id", "project_name", "project_type", "investment_budget", "actual_spend", "budget_variance", "total_expected_benefit", "total_realized_benefit", "benefit_realization_pct"]].head(8))

Technology Value Realization Portfolio Summary:


---

## 3. Portfolio-Level Benefit Realization Waterfall

We visualize total portfolio expected benefits against the unrealized benefit gap and net realized benefits:

In [2]:
tot_expected = tvr_df["total_expected_benefit"].sum() / 1e7
tot_realized = tvr_df["total_realized_benefit"].sum() / 1e7
tot_gap = tot_expected - tot_realized

fig = go.Figure(go.Waterfall(
    name="Benefit Realization",
    orientation="v",
    measure=["relative", "relative", "total"],
    x=["Total Expected Benefits", "Unrealized Benefit Gap", "Net Realized Benefits"],
    textposition="outside",
    text=[f"₹{tot_expected:.1f} Cr", f"-₹{tot_gap:.1f} Cr", f"₹{tot_realized:.1f} Cr"],
    y=[tot_expected, -tot_gap, tot_realized],
    connector=dict(line=dict(color="rgb(63, 63, 63)")),
    decreasing=dict(marker=dict(color="#d62728")),
    increasing=dict(marker=dict(color="#2ca02c")),
    totals=dict(marker=dict(color="#1f77b4"))
))

fig.update_layout(
    title="Enterprise Technology Portfolio: Benefit Realization Waterfall (₹ Crores)",
    showlegend=False,
    template="plotly_white"
)
fig.show()

---

## 4. Benefit Realization by Benefit Type

Benefits span 7 categories: Cost Reduction, Productivity, Revenue Enablement, Risk Reduction, Cycle Time Reduction, Capacity Release, and Customer Experience:

In [3]:
# Aggregate benefits by type
type_agg = db.query_df("""
    SELECT
        benefit_type,
        SUM(expected_value) as expected_val,
        SUM(realized_value) as realized_val,
        ROUND(SUM(realized_value) / NULLIF(SUM(expected_value), 0) * 100.0, 1) as realization_pct
    FROM benefits
    GROUP BY benefit_type
    ORDER BY expected_val DESC
""")

plt.figure(figsize=(10, 4.5))
x = np.arange(len(type_agg))
w = 0.35

plt.bar(x - w/2, type_agg["expected_val"] / 1e7, width=w, label="Expected Benefit (₹ Cr)", color="#2b5c8f")
plt.bar(x + w/2, type_agg["realized_val"] / 1e7, width=w, label="Realized Benefit (₹ Cr)", color="#2ca02c")

plt.xticks(x, type_agg["benefit_type"], rotation=30, ha="right", fontsize=9)
plt.ylabel("Benefit Value (₹ Crores)", fontsize=11)
plt.title("Benefit Realization Performance across Categories", fontsize=13)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---

## 5. Contrasting Scenario E (`PRJ003`) vs Scenario F (`PRJ014`)

We inspect the dossiers for the top-performing project vs the project with the largest benefit gap:

In [4]:
prj003_dossier = get_project_value_profile("PRJ003")
prj014_dossier = get_project_value_profile("PRJ014")

comp_projects = pd.DataFrame([
    {
        "Project ID": "PRJ003",
        "Project Name": prj003_dossier["project"]["project_name"],
        "Budget Variance (₹ Cr)": (prj003_dossier["project"]["actual_spend"] - prj003_dossier["project"]["investment_budget"]) / 1e7,
        "Expected Benefit (₹ Cr)": prj003_dossier["project"]["expected_annual_benefit"] / 1e7,
        "Realized Benefit (₹ Cr)": prj003_dossier["benefits"]["realized_value"].sum() / 1e7,
        "Realization %": f"{(prj003_dossier['benefits']['realized_value'].sum() / prj003_dossier['project']['expected_annual_benefit'])*100:.1f}%",
        "Status": "Exceeded Target (Scenario E)"
    },
    {
        "Project ID": "PRJ014",
        "Project Name": prj014_dossier["project"]["project_name"],
        "Budget Variance (₹ Cr)": (prj014_dossier["project"]["actual_spend"] - prj014_dossier["project"]["investment_budget"]) / 1e7,
        "Expected Benefit (₹ Cr)": prj014_dossier["project"]["expected_annual_benefit"] / 1e7,
        "Realized Benefit (₹ Cr)": prj014_dossier["benefits"]["realized_value"].sum() / 1e7,
        "Realization %": f"{(prj014_dossier['benefits']['realized_value'].sum() / prj014_dossier['project']['expected_annual_benefit'])*100:.1f}%",
        "Status": "Severe Benefit Gap (Scenario F)"
    }
])
display(comp_projects)

---

## 6. Grounding Benefits to Measurable KPI Outcomes

We inspect how project benefits connect to empirical KPI movements:

In [5]:
kpi_progression = analyze_benefit_kpi_progression()
print("Associated KPI Progression Table:")
display(kpi_progression[["project_id", "project_name", "benefit_type", "kpi_name", "baseline_value", "target_value", "actual_value", "unit", "kpi_target_achievement_pct"]].head(8))

Associated KPI Progression Table:


## 7. Limitations & Analytical Guardrails

1. **Observational Metrics:** Recorded realized benefits represent observed financial shifts in operational accounts. Without controlled econometric isolation, other macro factors (market growth, organizational restructuring) also influence outcomes.
2. **Language Guardrail:** Express outcomes using: *"The project is associated with a recorded realized benefit of..."* rather than declaring unverified direct causality.

## 8. Next Step

In **Notebook 10: Cost Driver & Variance Analysis**, we analyze monthly spending fluctuations, deconstructing spend spikes into healthy transaction scaling versus unbacked rate anomalies.

## 9. v2 Extensions: Financial Return Modeling, DiD Causal Inference, Stage Gates & Capability Maturity

Demonstrating:
1. **Capital Investment Financial Return Modeling:** Computing discounted cash flows, NPV, IRR, and payback periods across projects.
2. **Econometric Difference-in-Differences (DiD) Causal Inference:** Measuring treatment vs control unit cost divergences with consulting guardrails.
3. **Automated Portfolio Stage-Gate Health Scorecards:** Multi-criteria governance ratings across budget, schedule, and benefit risk.
4. **Capability Maturity Progression:** Tracking capability maturity uplift and transformation capital efficiency (₹/Maturity Point).


In [6]:
from tvi.value_realization import (
    calculate_investment_financial_metrics,
    estimate_causal_impact_did,
    generate_stage_gate_health_scorecard,
    evaluate_capability_maturity_progression,
)

print('=== 1. Capital Investment Financial Metrics (NPV & IRR) ===')
fin_metrics = calculate_investment_financial_metrics(discount_rate=0.08, time_horizon_years=5)
display(fin_metrics.head(10))

print('
=== 2. Econometric Difference-in-Differences (PRJ003 Modernization) ===')
did_eval = estimate_causal_impact_did('PRJ003')
for k, v in did_eval.items():
    print(f'  {k}: {v}')

print('
=== 3. Portfolio Stage-Gate Governance Scorecards ===')
scorecards = generate_stage_gate_health_scorecard()
display(scorecards.head(10))

print('
=== 4. Capability Maturity Progression & Transformation Impact ===')
mat_df = evaluate_capability_maturity_progression()
display(mat_df.head(10))


  Cell In[1], line 12
    print('
          ^
SyntaxError: unterminated string literal (detected at line 12)

